# Imports

In [ ]:
!pip install pyovf moviepy matplotlib -q


import os
import re
import glob
import math
import pyovf
import numpy as np
import matplotlib.pyplot as plt
from base64 import b64encode
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML, display
from moviepy.editor import ImageSequenceClip
from matplotlib.animation import FuncAnimation


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.3/110.3 kB 3.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



# Class to encode MuMax3 world

In [ ]:
class MuMaxWorld:
  def __init__(self, grid_dims=(128, 128, 1), cell_dims=(4e-9, 4e-9, 4e-9)):
    self.nx = grid_dims[0]
    self.ny = grid_dims[1]
    self.nz = grid_dims[2]
    self.dx = cell_dims[0]
    self.dy = cell_dims[1]
    self.dz = cell_dims[2]

  def get_grid_dims(self):
    return (self.nx, self.ny, self.nz)

  def get_cell_dims(self):
    return (self.dx, self.dy, self.dz)

  def compute_spatial_coordinates(self):
    x = (np.arange(self.nx) - (self.nx-1) / 2) * self.dx
    y = (np.arange(self.ny) - (self.ny-1) / 2) * self.dy
    z = (np.arange(self.nz) - (self.nz-1) / 2) * self.dz
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    self.X = X
    self.Y = Y
    self.Z = Z
    return X, Y, Z

# Class for OVF Generation

In [ ]:
class OVFGenerator:
  def __init__(self, mumax_world=None):
    self.__mumax_world = mumax_world
    self.__decay_axes = ('x', 'y')
    self.__field_axis = 'z'

  def __check_for_world(self):
    return (self.__mumax_world is not None)

  def set_decay_axes(self, new_decay_axes):
    self.__decay_axes = new_decay_axes

  def set_field_axis(self, new_field_axis):
    self.__field_axis = new_field_axis

  def set_mumax_world(self, mumax_world):
    self.__mumax_world = mumax_world

  def get_decay_axes(self):
    return self.__decay_axes

  def get_field_axis(self):
    return self.__field_axis

  def generate_radial_gaussian_ovf(self, amplitude, sigma, fname):
    if self.__check_for_world():
      X, Y, Z = self.__mumax_world.compute_spatial_coordinates()
      coord = {'x' : X, 'y' : Y, 'z' : Z}
      R2 = sum(coord[a]**2 for a in self.__decay_axes)
      mag = amplitude * np.exp(-R2 / (2 * sigma**2))
      Fx = mag if self.__field_axis == 'x' else np.zeros_like(mag)
      Fy = mag if self.__field_axis == 'y' else np.zeros_like(mag)
      Fz = mag if self.__field_axis == 'z' else np.zeros_like(mag)
      self.__write_to_ovf(Fx, Fy, Fz, fname)
    else:
      print('cannot create OVF for undefined MuMaxWorld')

  def generate_sincos_ovf(self, amplitude, wavelength, fname, phase=0.0,
                           use_cos=False):
    if self.__check_for_world():
      X, Y, Z = self.__mumax_world.compute_spatial_coordinates()
      coord = {'x': X, 'y': Y, 'z': Z}

      k = 2 * np.pi / wavelength
      arg = k * coord['x'] + phase
      mag = amplitude * (np.cos(arg) if use_cos else np.sin(arg))

      Fx = mag if self.__field_axis == 'x' else np.zeros_like(mag)
      Fy = mag if self.__field_axis == 'y' else np.zeros_like(mag)
      Fz = mag if self.__field_axis == 'z' else np.zeros_like(mag)
      self.__write_to_ovf(Fx, Fy, Fz, fname)
    else:
      print('cannot create OVF for undefined MuMaxWorld')

  def generate_windowed_sincos_ovf(self, amplitude, sigma, wavelength, fname, phase=0.0,
                                    use_cos=False):
    if self.__check_for_world():
      X, Y, Z = self.__mumax_world.compute_spatial_coordinates()
      coord = {'x': X, 'y': Y, 'z': Z}

      # Gaussian envelope over the configured decay axes
      R2 = sum(coord[a]**2 for a in self.__decay_axes)
      envelope = np.exp(-R2 / (2 * sigma**2))

      # Sinusoidal carrier along X
      k = 2 * np.pi / wavelength
      arg = k * coord['x'] + phase
      carrier = np.cos(arg) if use_cos else np.sin(arg)

      mag = amplitude * envelope * carrier

      Fx = mag if self.__field_axis == 'x' else np.zeros_like(mag)
      Fy = mag if self.__field_axis == 'y' else np.zeros_like(mag)
      Fz = mag if self.__field_axis == 'z' else np.zeros_like(mag)
      self.__write_to_ovf(Fx, Fy, Fz, fname)
    else:
      print('cannot create OVF for undefined MuMaxWorld')


  def __write_to_ovf(self, Fx, Fy, Fz, fname):
    nx = self.__mumax_world.nx
    ny = self.__mumax_world.ny
    nz = self.__mumax_world.nz
    dx = self.__mumax_world.dx
    dy = self.__mumax_world.dy
    dz = self.__mumax_world.dz
    with open(fname, "w") as f:
      f.write("# OOMMF OVF 2.0\n")
      f.write("# Segment count: 1\n")
      f.write("# Begin: Segment\n")
      f.write("# Begin: Header\n")
      f.write(f"# xnodes: {nx}\n")
      f.write(f"# ynodes: {ny}\n")
      f.write(f"# znodes: {nz}\n")
      f.write(f"# xstepsize: {dx}\n")
      f.write(f"# ystepsize: {dy}\n")
      f.write(f"# zstepsize: {dz}\n")
      f.write("# valuedim: 3\n")
      f.write("# End: Header\n")
      f.write("# Begin: Data text\n")

      # Standard OOMMF loop order (x moves fastest)
      for k in range(nz):
          for j in range(ny):
              for i in range(nx):
                  # FIX: Change back to [i, j, k] to match your (nx, ny, nz) shape
                  f.write(f"{Fx[i, j, k]:.6e} {Fy[i, j, k]:.6e} {Fz[i, j, k]:.6e}\n")

      f.write("# End: Data text\n")
      f.write("# End: Segment\n")

    print(f"Successfully generated '{fname}'.")


# Mumax Runner

In [ ]:
try:
    import google.colab
except ImportError:
    pass
else:
    # Download the mumax3 binary
    !wget https://mumax.ugent.be/mumax3-binaries/mumax3.12_linux_cuda12.9.tar.gz
    !tar -xvf mumax3.12_linux_cuda12.9.tar.gz
    !rm mumax3.12_linux_cuda12.9.tar.gz
    !rm -rf mumax3.12 && mv mumax3.12_linux_cuda12.9 mumax3.12
    #update the PATH environment variable
    import os
    os.environ['PATH'] += ":/content/mumax3.12"
    # Download an examplary script
    #!wget https://raw.githubusercontent.com/JeroenMulkers/mumax3-tutorial/master/standardproblem4.mx3 -O standardproblem4.mx3

--2026-06-30 22:01:03--  https://mumax.ugent.be/mumax3-binaries/mumax3.12_linux_cuda12.9.tar.gz
Resolving mumax.ugent.be (mumax.ugent.be)... 157.193.40.77
Connecting to mumax.ugent.be (mumax.ugent.be)|157.193.40.77|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 287329660 (274M) [application/x-gzip]
Saving to: ‘mumax3.12_linux_cuda12.9.tar.gz’

mumax3.12_linux_cud 100%[===================>] 274.02M  19.4MB/s    in 16s     

2026-06-30 22:01:20 (17.5 MB/s) - ‘mumax3.12_linux_cuda12.9.tar.gz’ saved [287329660/287329660]

mumax3.12_linux_cuda12.9/
mumax3.12_linux_cuda12.9/LICENSE
mumax3.12_linux_cuda12.9/mumax3
mumax3.12_linux_cuda12.9/mumax3-server
mumax3.12_linux_cuda12.9/mumax3-convert
mumax3.12_linux_cuda12.9/lib/
mumax3.12_linux_cuda12.9/lib/libcurand.so.10
mumax3.12_linux_cuda12.9/lib/libcufft.so.11


In [ ]:
def run_mumax3(filename):
  !mumax3 {filename}

In [ ]:
def convert_mumax_output(filename):
  !mumax3-convert -png {filename}/*.ovf

In [ ]:
def run_and_convert_mumax3(filename_wout_extension):
  fname_mx3 = f'{filename_wout_extension}.mx3'
  fname_out = f'{filename_wout_extension}.out'
  !mumax3 {fname_mx3}
  !mumax3-convert -png {fname_out}/*.ovf

In [ ]:
def parse_mx3(filename):
    """
    Parse a Mumax3 .mx3 file and return:
      - grid_size: (nx, ny, nz) from SetGridSize
      - cell_size: (dx, dy, dz) from SetCellSize
      - all other constants, keeping only the first value assigned
        (':=' or '='), evaluating expressions that reference
        previously defined variables (e.g. k0 := 2*pi/lambda0)
    """
    with open(filename, "r") as f:
        content = f.read()

    # Strip comments
    content = re.sub(r"//.*", "", content)

    params = {}

    # Grid size: SetGridSize(nx, ny, nz)
    m = re.search(r"SetGridSize\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)", content)
    if m:
        params["grid_size"] = tuple(int(v) for v in m.groups())

    # Cell size: SetCellSize(dx, dy, dz)
    m = re.search(r"SetCellSize\(\s*([\d.eE+-]+)\s*,\s*([\d.eE+-]+)\s*,\s*([\d.eE+-]+)\s*\)", content)
    if m:
        params["cell_size"] = tuple(float(v) for v in m.groups())

    # Safe namespace for eval: math functions/constants + collected vars
    safe_env = {
        "pi": math.pi, "e": math.e,
        "sin": math.sin, "cos": math.cos, "tan": math.tan,
        "sqrt": math.sqrt, "exp": math.exp, "log": math.log,
        "__builtins__": {},
    }

    # Generic assignments: name := expr   OR   name = expr (single line)
    pattern = re.compile(
        r"^\s*([A-Za-z_][A-Za-z0-9_]*)\s*:?=\s*(.+?)\s*$",
        re.MULTILINE
    )

    for match in pattern.finditer(content):
        key, expr = match.groups()
        if key in params:
            continue  # keep only the first occurrence

        # Skip anything that's clearly not a numeric expression
        # (function calls like SetGridSize(...), strings, etc.)
        if "(" in expr or '"' in expr or "'" in expr:
            continue

        try:
            value = eval(expr, safe_env, {**safe_env, **params})
            if isinstance(value, (int, float)):
                params[key] = float(value)
                safe_env[key] = float(value)  # make available to later lines
        except Exception:
            continue  # not a numeric expression we can resolve, skip silently

    return params

# Plotting


In [ ]:
class Visualizer:
    @staticmethod
    def __enforce_tuple(unk_input):
        if isinstance(unk_input, str):
            unk_input = (unk_input,)
        unk_input = tuple(ax.lower() for ax in unk_input)
        return unk_input

    @staticmethod
    def __open_ovf(filename, grid_dims):
      nx, ny, nz = grid_dims
      with open(filename, "rb") as f:
          for line in f:
              line_str = line.decode('utf-8', errors='ignore').strip()
              if "# Begin: Data Binary" in line_str:
                  break
          _ = np.fromfile(f, dtype=np.float32, count=1)
          total_nodes = nx * ny * nz
          raw_binary = np.fromfile(f, dtype=np.float32, count=total_nodes * 3)

      raw_vectors = raw_binary.reshape((nz, ny, nx, 3))
      spatial_vectors = np.transpose(raw_vectors, (2, 1, 0, 3))
      return spatial_vectors

    @staticmethod
    def __select_plotting_axes(decay_axes, grid_dims):
        nx, ny, nz = grid_dims
        dims = {'x': nx, 'y': ny, 'z': nz}
        all_axes = ['x', 'y', 'z']

        if len(decay_axes) == 2:
            h_axis, v_axis = decay_axes[0], decay_axes[1]
        elif len(decay_axes) == 1:
            h_axis = decay_axes[0]
            remaining = [ax for ax in all_axes if ax != h_axis]
            v_axis = remaining[0] if dims[remaining[0]] >= dims[remaining[1]] else remaining[1]
        else:
            h_axis, v_axis = 'x', 'y'

        axis_pair = (h_axis, v_axis)
        flat_axis = [ax for ax in all_axes if ax != h_axis and ax != v_axis][0]

        return axis_pair, flat_axis

    @staticmethod
    def __get_slice_idx(grid_dims, flat_axis):
        nx, ny, nz = grid_dims
        midpoints = {'x': nx // 2, 'y': ny // 2, 'z': nz // 2}
        slice_idx = midpoints[flat_axis]
        return slice_idx

    @staticmethod
    def __identify_axes_plotting_info(axis_pair, slice_idx):
        axis_key = "".join(sorted(axis_pair))
        full_slice = slice(None)

        plot_config = {
            "xy": {
                "slices": (full_slice, full_slice, slice_idx),
                "h_ax": "x", "v_ax": "y",
                "h_lbl": "X Position (nm)", "v_lbl": "Y Position (nm)"
            },
            "yz": {
                "slices": (slice_idx, full_slice, full_slice),
                "h_ax": "y", "v_ax": "z",
                "h_lbl": "Y Position (nm)", "v_lbl": "Z Position (nm)"
            },
            "xz": {
                "slices": (full_slice, slice_idx, full_slice),
                "h_ax": "x", "v_ax": "z",
                "h_lbl": "X Position (nm)", "v_lbl": "Z Position (nm)"
            }
        }
        cfg = plot_config[axis_key]
        return cfg

    @staticmethod
    def __setup_plot_based_on_axes(axis_pair, grid_dims, cell_dims, slice_idx, spatial_vectors):
        nx, ny, nz = grid_dims
        dx, dy, dz = cell_dims

        coords_1d = {
            'x': (np.arange(nx) - (nx - 1) / 2) * dx * 1e9,
            'y': (np.arange(ny) - (ny - 1) / 2) * dy * 1e9,
            'z': (np.arange(nz) - (nz - 1) / 2) * dz * 1e9
        }

        cfg = Visualizer.__identify_axes_plotting_info(axis_pair, slice_idx)

        Vx = spatial_vectors[cfg["slices"] + (0,)]
        Vy = spatial_vectors[cfg["slices"] + (1,)]
        Vz = spatial_vectors[cfg["slices"] + (2,)]

        # Generate grids and labels dynamically
        Grid_H, Grid_V = np.meshgrid(coords_1d[cfg["h_ax"]], coords_1d[cfg["v_ax"]], indexing='ij')
        h_label, v_label = cfg["h_lbl"], cfg["v_lbl"]

        return Vx, Vy, Vz, Grid_H, Grid_V, h_label, v_label

    @staticmethod
    def __render_plot(Vx, Vy, Vz, V_mag, Grid_H, Grid_V, h_label, v_label, field_axis, field_var, field_units):
        fig = plt.figure(figsize=(16, 12))
        components = [Vx, Vy, Vz, V_mag]
        titles = [
            f"Field Component ${field_var}_x$ (Field Axis: {field_axis})",
            f"Field Component ${field_var}_y$ (Field Axis: {field_axis})",
            f"Field Component ${field_var}_z$ (Field Axis: {field_axis})",
            f"Total Field Magnitude $|{field_var}|$"
        ]
        field_units_wrapped = ""
        if len(field_units) != 0:
            field_units_wrapped = f"({field_units})"
        for i in range(4):
            ax = fig.add_subplot(2, 2, i+1, projection='3d')
            surf = ax.plot_surface(Grid_H, Grid_V, components[i], cmap='viridis', edgecolor='none')

            ax.set_title(titles[i], fontsize=12)
            ax.set_xlabel(h_label)
            ax.set_ylabel(v_label)
            ax.set_zlabel(f"Field {field_units_wrapped}")
            fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label=field_units)

        plt.tight_layout()
        plt.show()

    @staticmethod
    def __setup_temp_image_folder():
      temp_img_folder = '/content/temp_frames'

      if os.path.exists(temp_img_folder):
        import shutil
        shutil.rmtree(temp_img_folder)
      os.makedirs(temp_img_folder, exist_ok=True)
      return temp_img_folder

    @staticmethod
    def __grab_ovf_files(folder_path, prefix_pattern):
      ovf_files = sorted(glob.glob(os.path.join(folder_path, f'{prefix_pattern}*.ovf')))
      if not ovf_files:
        print("No OVF files found. Check your file extension or path.")
        return
      print(f"Processing {len(ovf_files)} OVF files into uniform graphs...")
      return ovf_files

    @staticmethod
    def __generate_ovf_2D_graphs(ovf_files, temp_img_folder):
      import pyovf
      png_frames = []
      for i, ovf_path in enumerate(ovf_files):
        # Read the vector field data into a NumPy array
        ovf_obj = pyovf.read(ovf_path)
        data = ovf_obj.data

        # Take a 2D slice (adjust index depending on your simulation geometry)
        z_slice = data.shape[0] // 2
        mx = data[z_slice, :, :, 0] # X component

        # FIX: Explicitly set figure size
        fig, ax = plt.subplots(figsize=(6, 5))

        # FIX: vmin/vmax keeps the colorbar scale locked across the whole video
        im = ax.imshow(mx, cmap='RdBu', origin='lower', vmin=-1.0, vmax=1.0)
        ax.set_title(f"Frame {i}: Magnetization $M_x$")

        # FIX: Lock colorbar dimensions so it does not distort the frame size
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_aspect('equal')

        # FIX: Force strict margins instead of dynamic bounding boxes
        plt.tight_layout()

        # Save frame as a temporary PNG
        frame_path = os.path.join(temp_img_folder, f"frame_{i:06d}.png")
        plt.savefig(frame_path, dpi=150)
        plt.close(fig) # Prevent memory leaks in Colab

        png_frames.append(frame_path)
      return png_frames

    @staticmethod
    def __make_png_movie(png_files, output_name, fps=10):
      if not png_files:
          print("No matching PNG files found. Double-check your path.")
      else:
          print(f"Compiling video from {png_files[0]} to {png_files[-1]}...")
          clip = ImageSequenceClip(png_files, fps)

          # 4. Save video locally to the Colab session
          output_path = f'{output_name}.mp4'
          clip.write_videofile(output_path, codec='libx264', logger=None)

          # 5. Display the player directly in your browser
          mp4_data = open(output_path, 'rb').read()
          data_url = f"data:video/mp4;base64,{b64encode(mp4_data).decode()}"

          video_html = f"""
          <video width="640" controls autoplay loop>
              <source src="{data_url}" type="video/mp4">
          </video>
          """
          display(HTML(video_html))

    @staticmethod
    def get_plot_data(filename, grid_dims, cell_dims, decay_axes=('x', 'y')):
        decay_axes = Visualizer.__enforce_tuple(decay_axes)
        spatial_vectors = Visualizer.__open_ovf(filename, grid_dims)
        axis_pair, flat_axis = Visualizer.__select_plotting_axes(decay_axes, grid_dims)
        slice_idx = Visualizer.__get_slice_idx(grid_dims, flat_axis)
        Bx, By, Bz, Grid_H, Grid_V, h_label, v_label = Visualizer.__setup_plot_based_on_axes(
            axis_pair, grid_dims, cell_dims, slice_idx, spatial_vectors
        )
        B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)
        return Bx, By, Bz, Grid_H, Grid_V, B_mag, h_label, v_label

    @staticmethod
    def plot_field_ovf(filename, grid_dims, cell_dims, field_var='F', field_units='', decay_axes=('x', 'y'), field_axis='z'):
        Bx, By, Bz, Grid_H, Grid_V, B_mag, h_label, v_label = Visualizer.get_plot_data(
            filename, grid_dims, cell_dims, decay_axes=decay_axes
        )

        Visualizer.__render_plot(Bx, By, Bz, B_mag, Grid_H, Grid_V, h_label, v_label, field_axis, field_var, field_units)

    @staticmethod
    def display_ovf_2D_graph_movie(folder_path, prefix_pattern, fps=10):
      temp_img_folder = Visualizer.__setup_temp_image_folder()

      ovf_files = Visualizer.__grab_ovf_files(folder_path, prefix_pattern)

      if not ovf_files:
        return

      png_frames = Visualizer.__generate_ovf_2D_graphs(ovf_files, temp_img_folder)

      if not png_frames:
        return

      print("\nStitching graphs into a video...")
      Visualizer.__make_png_movie(png_frames, 'ovf_simulation_movie', fps)

    @staticmethod
    def make_png_movie_from_folder(folder_path, output_name, pattern='m', fps=10):
      png_files = sorted(glob.glob(os.path.join(folder_path, f'{pattern}*.png')))
      Visualizer.__make_png_movie(png_files, output_name, fps=fps)

    @staticmethod
    def animate_ovf_series(
            grid_dims,
            cell_dims,
            pattern="B_ext*.ovf",
            output="gaussian_pulse.mp4",
            decay_axes=('x', 'y'),
            fps=20):
        """
        Animate a time series of OVF files using the same loading/slicing
        pipeline as get_plot_data / plot_field_ovf, so no separate
        loader (e.g. load_generalized_ovf) is needed.
        """
        files = sorted(glob.glob(pattern))

        if len(files) == 0:
            raise FileNotFoundError(f"No files matched pattern {pattern}")

        print(f"Found {len(files)} OVF files")

        # ----------------------------------------
        # Load every frame ONCE up front. The original version read each
        # file from disk twice (once to find max_field, once again inside
        # update() during animation) -- caching removes that redundant I/O
        # and redundant slicing/meshgrid work entirely.
        # ----------------------------------------
        frames_data = [
            Visualizer.get_plot_data(f, grid_dims, cell_dims, decay_axes=decay_axes)
            for f in files
        ]

        # h_label/v_label are identical across frames (grid_dims/decay_axes
        # don't change between files), so just grab them once.
        h_label, v_label = frames_data[0][6], frames_data[0][7]

        max_field = max(
            max(np.max(np.abs(Bx)), np.max(np.abs(By)), np.max(np.abs(Bz)), np.max(Bmag))
            for Bx, By, Bz, _, _, Bmag, _, _ in frames_data
        )

        fig = plt.figure(figsize=(16, 12))
        titles = [r"$B_x$", r"$B_y$", r"$B_z$", r"$|B|$"]

        def update(frame):
            fig.clear()

            Bx, By, Bz, Grid_H, Grid_V, Bmag, _, _ = frames_data[frame]
            data = [Bx, By, Bz, Bmag]

            for i in range(4):
                ax = fig.add_subplot(2, 2, i + 1, projection='3d')
                ax.plot_surface(Grid_H, Grid_V, data[i], cmap='viridis', edgecolor='none')
                ax.set_zlim(0, max_field)
                ax.set_title(f"{titles[i]}   Frame {frame}")
                ax.set_xlabel(h_label)
                ax.set_ylabel(v_label)
                ax.set_zlabel("Field (T)")

            plt.tight_layout()

        ani = FuncAnimation(fig, update, frames=len(files), interval=50)
        ani.save(output, writer='ffmpeg', fps=fps)

        print(f"Saved movie: {output}")

# Adjusting Bfield

In [ ]:
file_name = "YIGMagnon_mini"

In [ ]:
mumax_world_values = parse_mx3(f"{file_name}.mx3")
for k, v in mumax_world_values.items():
    print(f"{k}: {v}")
mumax_world = MuMaxWorld(mumax_world_values['grid_size'], mumax_world_values['cell_size'])
ovf_generator = OVFGenerator(mumax_world)


In [ ]:
ovf_generator.set_field_axis('x')
ovf_generator.generate_windowed_sincos_ovf(amplitude=mumax_world_values['amp_p'], sigma=mumax_world_values['sigma_p'], wavelength=mumax_world_values['lambda0'], fname="bprofile_sin.ovf", use_cos=False)
ovf_generator.generate_windowed_sincos_ovf(amplitude=mumax_world_values['amp_p'], sigma=mumax_world_values['sigma_p'], wavelength=mumax_world_values['lambda0'], fname="bprofile_cos.ovf", use_cos=True)

In [ ]:
run_mumax3(f'{file_name}.mx3')

In [ ]:
Visualizer.animate_ovf_series(
            mumax_world_values['grid_size'],
            mumax_world_values['cell_size'],
            pattern=f"/content/{file_name}.out/m*.ovf",
            output="yig_magnon_m.mp4",
            decay_axes=('x', 'y'),
            fps=10)

In [ ]:
Visualizer.animate_ovf_series(
            mumax_world_values['grid_size'],
            mumax_world_values['cell_size'],
            pattern=f"/content/{file_name}.out/B_ext*.ovf",
            output="yig_magnon_b.mp4",
            decay_axes=('x', 'y'),
            fps=10)